In [29]:
# Define source path for journal results
SOURCE_PATH = "../../save_and_results/old/journal/"

In [34]:
import sys
sys.path.append('..')

import pickle
import numpy as np
from matplotlib.pyplot import cm
import pandas as pd
from numpy import linspace
import matplotlib.pyplot as plt

from utils.display_tools import load_best_forecasts,  display_predictions_2
from os import listdir
from os.path import isfile, join
from utils.general_tools import results_to_pd

In [35]:
# TODO SOMETHNG IS OFF HERE WITH THE SEARCH SPACE (Some combinations are missing. Find out what happens here and fix it for the revision.)

In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
lot = pickle.load(open("../../save_and_results/cache_lot.p", "rb"))
data = pickle.load(open("../../save_and_results/cache_endog.p", "rb"))
exogs = pickle.load(open("../../save_and_results/cache_exogs.p", "rb"))

In [38]:
example_id = "10905"

In [39]:
methods = ["ada", "forest", "linear", "sarimax", "var"]

In [40]:
full_stack = []
for m in methods:
    res = []
    for x in ["", "_univariate","_remove_T","_remove_W"]:
        try:
            res.append(results_to_pd(pickle.load(open(SOURCE_PATH + "journal_new" + x + "/Moehne_desc1_" + m + "_results_stack.p", "rb"))))
        except:
            print("Error: " + m + x)
            continue
    res = pd.concat(res)
    res = res.loc[res["Index"] == example_id]
    res["method"] = m
    full_stack.append(res)

Error: var_univariate


In [41]:
full_stack = pd.concat(full_stack)[['P', 'D', 'Q', 'STAU', 'T',
       'M_Stau', 'M_T', 'Decompose', 'Interaction', 'Estimator', 'n_estimator',
       'max_depth', 'learning_rate', 'method']].reset_index(drop=True)

/tmp/ipykernel_99578/1737725031.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_stack = pd.concat(full_stack)[['P', 'D', 'Q', 'STAU', 'T',


In [42]:
full_stack["STAU"] = full_stack["STAU"].astype(str)
full_stack["T"] =full_stack["T"].astype(str)

In [43]:
final = []
for method_name in full_stack["method"].unique():
    method = full_stack[full_stack["method"] == method_name]
    method = method[[x for x in method.columns if len(method[x].unique()) > 1]]
    method = pd.DataFrame([[list(method[x].unique())] for x in method.columns], index = method.columns, columns=[method_name])
    final.append(method)

In [44]:
search = pd.concat(final, axis=1)
search[search.isnull()] = "-"


In [45]:
search[search.isnull()] = "-"


In [53]:
search.loc["STAU"] = "[-,0,1,2]"
search.loc["T"] = "[-,0,1,2]"
search.loc["max_depth", "forest"] = '["-", 3, 7]'
search.loc["learning_rate", "ada"] = '[1, 0.1]'
search = search[["linear", "sarimax", "var", "forest", "ada"]]

In [54]:
search 


,linear,sarimax,var,forest,ada
P,"[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]"
STAU,"[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]"
T,"[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]"
M_Stau,"[0, 7]","[0, 7]","[0, 7]","[0, 7]","[0, 7]"
M_T,"[0, 7]","[0, 7]","[0, 7]","[0, 7]","[0, 7]"
Decompose,"[0, 1]","[0, 1]","[0, 1]","[0, 1]","[0, 1]"
Interaction,"[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]"
Estimator,-,-,-,-,"[dt, linear]"
n_estimator,-,-,-,"[50, 250]","[50, 250]"
learning_rate,-,-,-,-,"[1, 0.1]"


In [55]:
print(search.to_latex())

\begin{tabular}{llllll}
\toprule
 & linear & sarimax & var & forest & ada \\
\midrule
P & [np.int64(0), np.int64(1), np.int64(2)] & [np.int64(0), np.int64(1), np.int64(2)] & [np.int64(0), np.int64(1), np.int64(2)] & [np.int64(0), np.int64(1), np.int64(2)] & [np.int64(0), np.int64(1), np.int64(2)] \\
STAU & [-,0,1,2] & [-,0,1,2] & [-,0,1,2] & [-,0,1,2] & [-,0,1,2] \\
T & [-,0,1,2] & [-,0,1,2] & [-,0,1,2] & [-,0,1,2] & [-,0,1,2] \\
M_Stau & [np.int64(0), np.int64(7)] & [np.int64(0), np.int64(7)] & [np.int64(0), np.int64(7)] & [np.int64(0), np.int64(7)] & [np.int64(0), np.int64(7)] \\
M_T & [np.int64(0), np.int64(7)] & [np.int64(0), np.int64(7)] & [np.int64(0), np.int64(7)] & [np.int64(0), np.int64(7)] & [np.int64(0), np.int64(7)] \\
Decompose & [np.int64(0), np.int64(1)] & [np.int64(0), np.int64(1)] & [np.int64(0), np.int64(1)] & [np.int64(0), np.int64(1)] & [np.int64(0), np.int64(1)] \\
Interaction & [np.int64(0), np.int64(1), np.int64(2)] & [np.int64(0), np.int64(1), np.int64(2)] & [np

In [56]:
c = full_stack[full_stack["method"] == "linear"]
c = c[[x for x in c.columns if len(c[x].unique()) > 1]]
c.loc[(c["T"] == "[]") & (c["STAU"] == "[]")]

,P,STAU,T,M_Stau,M_T,Decompose,Interaction
15824,1,[],[],0,0,0,0
15825,1,[],[],0,0,1,0
15826,2,[],[],0,0,0,0
15827,2,[],[],0,0,1,0


In [57]:
c.loc[(c["T"] == "[]") & (c["STAU"] == "[]")]

,P,STAU,T,M_Stau,M_T,Decompose,Interaction
15824,1,[],[],0,0,0,0
15825,1,[],[],0,0,1,0
15826,2,[],[],0,0,0,0
15827,2,[],[],0,0,1,0


In [61]:
search["linear"].dropna()

P                [0, 1, 2]
STAU             [-,0,1,2]
T                [-,0,1,2]
M_Stau              [0, 7]
M_T                 [0, 7]
Decompose           [0, 1]
Interaction      [0, 1, 2]
Estimator                -
n_estimator              -
learning_rate            -
max_depth                -
D                        -
Q                        -
Name: linear, dtype: object

In [59]:
3 * 4 * 4 *2 * 2 * 2 * 3 * 2 * 2 * 2

9216